# PathSnap dataset exploration

Starter EDA for trip inventory, GPS sampling intervals, and road-network sizes. The notebook reads only `train` and `student_test`; keep evaluator data separate. `.route` values are reference routes and should not be described as independently verified ground truth.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.parser import discover_trips, parse_trip
from src.features import trip_feature_summary

RAW_ROOT = Path('../dataset_raw/Student folder')


In [ ]:
records = []
for split in ('train', 'student_test'):
    for folder in discover_trips(RAW_ROOT / split):
        trip = parse_trip(folder, require_route=True)
        row = trip_feature_summary(trip)
        row['split'] = split
        records.append(row)
inventory = pd.DataFrame(records)
inventory.groupby('split')[['gps_observations', 'node_count', 'arc_count', 'route_arc_count']].describe()

In [ ]:
inventory.boxplot(column='gps_observations', by='split')
plt.title('GPS observations per trip')
plt.suptitle('')
plt.ylabel('Observations')
plt.show()

## Inspect a trip

Review missing values, timestamp gaps, geographic extent, network connectivity, and route coverage before interpreting candidate model scores. A Folium map can be added here when the optional visualization dependency is installed.

In [ ]:
trip = parse_trip(next(iter(discover_trips(RAW_ROOT / 'train'))))
track = trip.track
track[['longitude', 'latitude']].plot(x='longitude', y='latitude', style='.-',
                                      title=f'GPS trace: {trip.trip_id}',
                                      figsize=(7, 6), legend=False)
plt.axis('equal')
plt.show()
track.timestamp.diff().describe()